In [ ]:
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input,LSTM,Embedding,Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
eng = ['hi','hello','how are you','thank you']
fr = ['salut','bonjour','comment ca va','merci']

In [ ]:
#tokenize english
tok_eng = Tokenizer()
tok_eng.fit_on_texts(eng)
x = tok_eng.texts_to_sequences(eng)
x = pad_sequences(x)


In [ ]:
x

array([[0, 0, 2],
       [0, 0, 3],
       [4, 5, 1],
       [0, 6, 1]], dtype=int32)

In [ ]:
#tokenize french
tok_fr = Tokenizer()
tok_fr.fit_on_texts(fr)
y = tok_fr.texts_to_sequences(fr)
y = pad_sequences(y)


In [ ]:
y

array([[0, 0, 1],
       [0, 0, 2],
       [3, 4, 5],
       [0, 0, 6]], dtype=int32)

In [ ]:
y_in = y[:,:-1]
y_out = y[:,1:]

In [ ]:
y_in

array([[0, 0],
       [0, 0],
       [3, 4],
       [0, 0]], dtype=int32)

In [ ]:
y_out

array([[0, 1],
       [0, 2],
       [4, 5],
       [0, 6]], dtype=int32)

In [ ]:
#lstm - 3d - (samples,timesteps,vocab_size)
# - our labels must be in 3d shape as well
#3d - adding 1 to it at the end

In [ ]:
y_out = y_out.reshape((y_out.shape[0],y_out.shape[1],1))

In [ ]:
y_out

array([[[0],
        [1]],

       [[0],
        [2]],

       [[4],
        [5]],

       [[0],
        [6]]], dtype=int32)

In [ ]:
vocab_eng = len(tok_eng.word_index)+1
vocab_fr = len(tok_fr.word_index)+1

why 8 - each word is now a 8-dim menaing vector

In [ ]:
#encoder
enc_in =Input(shape =(x.shape[1],)) #each english input seq
enc_emb = Embedding(vocab_eng,8)(enc_in) #learn word embeddings
_,h,c  = LSTM(32,return_state= True)(enc_emb)


In [ ]:
#decoder

dec_in= Input(shape =(y_in.shape[1],))
dec_emb = Embedding(vocab_fr,8)(dec_in)
dec_out = LSTM(32, return_sequences=True)(dec_emb,initial_state =[h,c])
out = Dense(vocab_fr ,activation= 'softmax')(dec_out)

In [ ]:
model = Model([enc_in,dec_in],out)
model.compile(optimizer = 'adam',loss ='sparse_categorical_crossentropy')

In [ ]:
#train
model.fit([x,y_in],y_out,epochs = 300)

Epoch 1/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - loss: 1.9463
Epoch 2/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - loss: 1.9439
Epoch 3/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - loss: 1.9415
Epoch 4/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - loss: 1.9392
Epoch 5/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - loss: 1.9368
Epoch 6/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - loss: 1.9344
Epoch 7/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - loss: 1.9319
Epoch 8/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - loss: 1.9295
Epoch 9/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - loss: 1.9270
Epoch 10/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - loss: 1.9244
Epoch 11/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - loss: 1.9217
Epoch 12/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - loss: 1.9190
Epoch 13/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - loss: 1.9163
Epoch 14/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - loss: 1.9134
Epoch 15/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - loss: 1.9104
Epoch 16/300
1/1 ━━━━

In [ ]:
#predict

preds = model.predict([x,y_in])
for i,pred in enumerate(preds):
  ids = np.argmax(pred,axis = 1)
  words = [tok_fr.index_word.get(idx, "??") for idx in ids]
  print(f"English : {eng[i]} ----> pred french: {' '.join(words)}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
English : hi ----> pred french: ???????? salut
English : hello ----> pred french: ???????? bonjour
English : how are you ----> pred french: ca va
English : thank you ----> pred french: ???????? merci


In [ ]:
#return_state ---> give h,c state as well along with output
#resturn_sequences ---> tell LSTM to output at every time step ,not just at the end